In [1]:
import xgboost as xgb
import lightgbm as lgb
print(f"XGBoost:  {xgb.__version__}")
print(f"LightGBM: {lgb.__version__}")

XGBoost:  3.2.0
LightGBM: 4.6.0


---
# **Q.  XGBClassifier: key parameters**
---


In [2]:
import time
import numpy as np
from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

xgb_model = XGBClassifier(
    n_estimators     = 200,   # boosting rounds
    learning_rate    = 0.1,   # how much each tree contributes
    max_depth        = 4,     # tree depth (try 3-8)
    subsample        = 0.8,   # fraction of ROWS per tree
    colsample_bytree = 0.8,   # fraction of COLUMNS per tree
    reg_alpha        = 0,     # L1 on leaf weights
    reg_lambda       = 1,     # L2 on leaf weights (default)
    use_label_encoder=False,
    eval_metric      = 'logloss',
    verbosity        = 0,
    random_state     = 42
)

t0 = time.time()
xgb_model.fit(X_train, y_train)
print(f"XGBoost train: {xgb_model.score(X_train,y_train):.4f}  "
      f"test: {xgb_model.score(X_test,y_test):.4f}  "
      f"time: {time.time()-t0:.3f}s")

print("\nParameter roles:")
print("  n_estimators     → more = better until overfit")
print("  learning_rate    → smaller = slower but often better")
print("  max_depth        → deeper = more complex")
print("  colsample_bytree → like max_features in RF (diversity)")
print("  reg_lambda       → L2 keeps leaf weights smooth")

XGBoost train: 1.0000  test: 0.9649  time: 0.203s

Parameter roles:
  n_estimators     → more = better until overfit
  learning_rate    → smaller = slower but often better
  max_depth        → deeper = more complex
  colsample_bytree → like max_features in RF (diversity)
  reg_lambda       → L2 keeps leaf weights smooth


#Insights:
---
| Parameter                     | Meaning                                                         | Effect                                                                                       |
| ----------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------------------------------------- |
| **n_estimators = 200**        | Number of decision trees (boosting rounds).                     | More trees can improve accuracy but increase training time and may overfit if too large.     |
| **learning_rate = 0.1**       | Shrinks the contribution of each new tree.                      | Smaller values learn more slowly and usually require more trees but often generalize better. |
| **max_depth = 4**             | Maximum depth of each decision tree.                            | Larger depth captures more complex patterns but increases overfitting risk.                  |
| **subsample = 0.8**           | Fraction of training **rows** randomly used to build each tree. | Reduces overfitting by making trees different from one another.                              |
| **colsample_bytree = 0.8**    | Fraction of **features (columns)** randomly used for each tree. | Prevents reliance on a few features and improves generalization.                             |
| **reg_alpha = 0**             | L1 regularization on leaf weights.                              | Encourages sparsity; larger values can reduce overfitting by driving some weights to zero.   |
| **reg_lambda = 1**            | L2 regularization on leaf weights.                              | Penalizes large weights, making the model more stable and reducing overfitting.              |
| **use_label_encoder = False** | Disables the old built-in label encoder.                        | Modern versions expect labels to already be encoded; this avoids a deprecation warning.      |
| **eval_metric = 'logloss'**   | Metric used to evaluate training performance.                   | `'logloss'` is commonly used for binary classification.                                      |
| **verbosity = 0**             | Controls console output during training.                        | `0` = silent, `1` = warnings, `2` = info, `3` = debug.                                       |
| **random_state = 42**         | Random seed for reproducibility.                                | Ensures the same results every time you run the code.                                        |


---
# **Q. LightGBM: leaf-wise growth**
---


In [3]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators     = 200,
    learning_rate    = 0.1,
    max_depth        = -1,   # -1 = no limit (controlled by num_leaves)
    num_leaves       = 31,   # KEY PARAM — max leaves per tree
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    verbose          = -1
)

t0 = time.time()
lgb_model.fit(X_train, y_train)
print(f"LightGBM train: {lgb_model.score(X_train,y_train):.4f}  "
      f"test: {lgb_model.score(X_test,y_test):.4f}  "
      f"time: {time.time()-t0:.3f}s")

print("\nLevel-wise (XGBoost) vs Leaf-wise (LightGBM):")
print("  XGBoost:  grows ALL nodes at current depth before going deeper")
print("            → balanced, robust, slower on big data")
print("  LightGBM: always grows the ONE leaf with highest loss reduction")
print("            → asymmetric, faster convergence, needs num_leaves cap")

LightGBM train: 1.0000  test: 0.9649  time: 0.268s

Level-wise (XGBoost) vs Leaf-wise (LightGBM):
  XGBoost:  grows ALL nodes at current depth before going deeper
            → balanced, robust, slower on big data
  LightGBM: always grows the ONE leaf with highest loss reduction
            → asymmetric, faster convergence, needs num_leaves cap


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


---
# **Q.  full benchmark: all four models**
---


In [4]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

all_models = {
    'sklearn GBM':  GradientBoostingClassifier(
                        n_estimators=200, learning_rate=0.1,
                        max_depth=3, random_state=42),
    'XGBoost':      XGBClassifier(
                        n_estimators=200, learning_rate=0.1, max_depth=4,
                        subsample=0.8, colsample_bytree=0.8,
                        use_label_encoder=False, eval_metric='logloss',
                        verbosity=0, random_state=42),
    'LightGBM':     lgb.LGBMClassifier(
                        n_estimators=200, learning_rate=0.1, num_leaves=31,
                        subsample=0.8, colsample_bytree=0.8,
                        random_state=42, verbose=-1),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42),
}

print(f"{'Model':20s}  {'Train':>8}  {'Test':>8}  {'Time':>8}")
print("-" * 50)
for name, m in all_models.items():
    t0 = time.time()
    m.fit(X_train, y_train)
    elapsed = time.time() - t0
    print(f"{name:20s}  {m.score(X_train,y_train):8.4f}  "
          f"{m.score(X_test,y_test):8.4f}  {elapsed:7.3f}s")

Model                    Train      Test      Time
--------------------------------------------------
sklearn GBM             1.0000    0.9532    4.070s
XGBoost                 1.0000    0.9649    1.011s
LightGBM                1.0000    0.9649    0.119s


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


RandomForest            1.0000    0.9415    0.508s


---
# **Q. Vary XGBoost learning_rate (0.001, 0.01, 0.05, 0.1, 0.3, 1.0) with n_estimators=200. What is the best Ir and what happens at extremes?**
---


In [5]:
from xgboost import XGBClassifier

print(f"{'lr':>7}  {'train':>8}  {'test':>8}  {'verdict':>28}")
print("-" * 57)
for lr in [0.001, 0.01, 0.05, 0.1, 0.3, 1.0]:
    xgb = XGBClassifier(n_estimators=200, learning_rate=lr, max_depth=4,
                        subsample=0.8, colsample_bytree=0.8,
                        use_label_encoder=False, eval_metric='logloss',
                        verbosity=0, random_state=42)
    xgb.fit(X_train, y_train)
    tr = xgb.score(X_train, y_train)
    te = xgb.score(X_test,  y_test)
    v = "underfit — needs more trees" if te<0.93 else ("overfit" if tr-te>0.04 else "good")
    print(f"{lr:7.3f}  {tr:8.4f}  {te:8.4f}  {v:>28}")

     lr     train      test                       verdict
---------------------------------------------------------
  0.001    0.6281    0.6257   underfit — needs more trees
  0.010    0.9925    0.9415                       overfit
  0.050    1.0000    0.9591                       overfit
  0.100    1.0000    0.9649                          good
  0.300    1.0000    0.9649                          good
  1.000    1.0000    0.9591                       overfit


---
# **Q. Compare colsample_bytree=1.0 vs 0.8 vs 0.5 vs 0.3. Is the pattern the same as max_features in Random Forest?**
---


In [6]:
print(f"{'colsample':>10}  {'train':>8}  {'test':>8}")
print("-" * 32)
for csb in [1.0, 0.8, 0.5, 0.3, 0.1]:
    xgb = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4,
                        subsample=0.8, colsample_bytree=csb,
                        use_label_encoder=False, eval_metric='logloss',
                        verbosity=0, random_state=42)
    xgb.fit(X_train, y_train)
    print(f"{csb:10.1f}  {xgb.score(X_train,y_train):8.4f}  "
          f"{xgb.score(X_test,y_test):8.4f}")

 colsample     train      test
--------------------------------
       1.0    1.0000    0.9649
       0.8    1.0000    0.9649
       0.5    1.0000    0.9649
       0.3    1.0000    0.9532
       0.1    1.0000    0.9532


#Insights:
---
>Yes, same pattern as Random Forest's max_features. Using 100% columns is slightly worse (although not in this case) for the dataset which have heavy reliance on a single feature which will lead to ignoring the impact of other features — forced feature diversity reduces tree correlation, reducing ensemble variance. Sweet spot is 0.5-0.8. At 0.1 (only 3/30 features) individual trees are too weak — too much diversity hurts.

Yes! It is **very similar to `max_features` in Random Forest**. The easiest way to understand it is with a simple example.

---

## Imagine you have a dataset

Suppose your dataset has **10 features**.

| Feature        |
| -------------- |
| Age            |
| Salary         |
| Education      |
| Experience     |
| Credit Score   |
| Loan Amount    |
| Marital Status |
| Gender         |
| City           |
| Savings        |

When XGBoost builds **one tree**, `colsample_bytree` determines **how many of these features the tree is allowed to use**.

---

# Case 1: `colsample_bytree = 1.0`

Every tree gets **all 10 features**.

```
Tree 1:
Age
Salary
Education
Experience
Credit Score
Loan Amount
Marital Status
Gender
City
Savings

Tree 2:
Age
Salary
Education
Experience
Credit Score
Loan Amount
Marital Status
Gender
City
Savings
```

Now imagine **Credit Score** is an extremely powerful feature.

Every tree discovers

```
Credit Score
      ↓
 Best Split
```

So almost every tree starts looking similar.

Example

```
Tree 1
Credit Score > 700 ?

Tree 2
Credit Score > 710 ?

Tree 3
Credit Score > 690 ?
```

The ensemble lacks diversity.

---

# Case 2: `colsample_bytree = 0.8`

Now each tree only sees **8 of the 10 features**.

Example

```
Tree 1
Age
Salary
Education
Experience
Credit Score
Loan
City
Savings

Tree 2
Salary
Education
Experience
Loan
Gender
City
Savings
Marital Status

(Tree 2 doesn't even have Credit Score!)
```

Since some trees don't have access to the strongest feature, they must learn from other features.

Example

```
Tree 1
Credit Score

Tree 2
Salary

Tree 3
Experience

Tree 4
Savings
```

Now every tree learns something slightly different.

When you combine them,

```
Prediction =
Average(Tree1, Tree2, Tree3...)
```

the overall model becomes more robust and less likely to overfit.

---

# Case 3: `colsample_bytree = 0.5`

Each tree only gets **5 features**.

Example

```
Tree 1
Age
Salary
Loan
City
Savings

Tree 2
Education
Gender
Experience
Loan
Credit Score

Tree 3
Age
Education
Savings
Gender
City
```

Now diversity becomes even larger.

Many datasets actually perform best around

```
0.5 - 0.8
```

because trees are different enough while still having enough information.

---

# Case 4: `colsample_bytree = 0.3`

Each tree only gets **3 features**.

Example

```
Tree 1
Age
Gender
City

Tree 2
Loan
Savings
Experience

Tree 3
Salary
Education
Marital Status
```

Now many important features are missing.

Maybe Tree 1 never even sees

```
Credit Score
```

which is the most informative feature.

That tree becomes weak.

If every tree is weak,

the whole ensemble also becomes weaker.

---

# Why not always use 1.0?

Suppose

```
Credit Score
```

alone predicts almost everything.

If every tree always sees it,

every tree becomes

```
Credit Score
      ↓
almost identical tree
```

Imagine 100 students taking an exam.

If everyone copies from the same smartest student,

all mistakes are identical.

There is **no benefit** in averaging their answers.

But if students solve the exam independently,

some make different mistakes.

When you combine their answers,

the overall result improves.

That's exactly what happens with feature sampling.

---

# Relation to Random Forest

Random Forest uses

```python
max_features
```

XGBoost uses

```python
colsample_bytree
```

Both do the same basic job:

> **Randomly select only a subset of features for each tree, increasing diversity and reducing overfitting.**

The difference is **when** they are used:

* **Random Forest:** Trees are built independently and averaged. `max_features` makes those independent trees less correlated.
* **XGBoost:** Trees are built sequentially to correct previous errors. `colsample_bytree` still introduces feature diversity, helping prevent later trees from overfitting to the same dominant features.

---

## Easy visualization

```
colsample = 1.0

Tree1 → Uses all features
Tree2 → Uses all features
Tree3 → Uses all features

Trees become very similar
↑ Higher correlation
↑ More overfitting


colsample = 0.8

Tree1 → 80% features
Tree2 → Different 80%
Tree3 → Different 80%

Trees become different
✓ Better diversity
✓ Better generalization


colsample = 0.3

Tree1 → Only 3 features
Tree2 → Different 3 features
Tree3 → Different 3 features

Trees become too weak
↓ Accuracy
```

**The key idea:** `colsample_bytree` is about balancing **diversity** and **strength**. Too high (1.0) can make trees overly similar; too low (e.g., 0.3) can make each tree too weak because it doesn't have enough useful information. Values around **0.5–0.8** often provide a good balance.


---
# **Q. Explain the difference between max_depth (XGBoost) and num_leaves (LightGBM). Show with code that num_leaves ≈ 2^max_depth.**
---


In [7]:
import lightgbm as lgb, numpy as np

print("Level-wise (XGBoost): max_depth=d → at most 2^d leaves")
print("Leaf-wise (LightGBM): num_leaves=N → N leaves, asymmetric depth")
print()
print(f"{'XGB max_depth':>14}  {'Max leaves':>11}  {'Use num_leaves':>14}")
for d in [3, 4, 5, 6, 7]:
    print(f"{d:14d}  {2**d:11d}  {2**d:14d}")

print("\n--- Same leaf count, XGBoost style vs LightGBM style ---")
for nl in [8, 16, 31, 63, 127]:
    lgb_m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1,
                                num_leaves=nl, subsample=0.8,
                                colsample_bytree=0.8, random_state=42, verbose=-1)
    lgb_m.fit(X_train, y_train)
    d_equiv = int(np.log2(nl))
    print(f"num_leaves={nl:4d}  (~max_depth≈{d_equiv})  "
          f"test={lgb_m.score(X_test,y_test):.4f}")

Level-wise (XGBoost): max_depth=d → at most 2^d leaves
Leaf-wise (LightGBM): num_leaves=N → N leaves, asymmetric depth

 XGB max_depth   Max leaves  Use num_leaves
             3            8               8
             4           16              16
             5           32              32
             6           64              64
             7          128             128

--- Same leaf count, XGBoost style vs LightGBM style ---
num_leaves=   8  (~max_depth≈3)  test=0.9649


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=  16  (~max_depth≈4)  test=0.9649
num_leaves=  31  (~max_depth≈4)  test=0.9649


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=  63  (~max_depth≈5)  test=0.9649
num_leaves= 127  (~max_depth≈6)  test=0.9649


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


#Insights:
---
>Translation rule: set num_leaves ≤ 2^(XGB max_depth). LightGBM's leaf-wise growth often achieves the same accuracy with fewer leaves than the equivalent XGBoost tree — it allocates complexity where it's most needed rather than evenly across all branches.

---
# **Q. Run both XGBoost and LightGBM on the California Housing dataset (regression). Compare RMSE and training time.**
---


In [8]:
import time, numpy as np
from xgboost import XGBRegressor
import lightgbm as lgb
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

cal = fetch_california_housing()
X_c, y_c = cal.data, cal.target
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)
print(f"Dataset: {X_c.shape[0]:,} rows × {X_c.shape[1]} features\n")

models_r = {
    'sklearn GBM': GradientBoostingRegressor(
                       n_estimators=300, learning_rate=0.1,
                       max_depth=4, random_state=42),
    'XGBoost':     XGBRegressor(
                       n_estimators=300, learning_rate=0.1, max_depth=5,
                       subsample=0.8, colsample_bytree=0.8,
                       verbosity=0, random_state=42),
    'LightGBM':    lgb.LGBMRegressor(
                       n_estimators=300, learning_rate=0.1, num_leaves=63,
                       subsample=0.8, colsample_bytree=0.8,
                       random_state=42, verbose=-1),
    'RandomForest':RandomForestRegressor(n_estimators=200, random_state=42),
}

print(f"{'Model':20s}  {'RMSE':>8}  {'R²':>8}  {'Time':>8}")
print("-" * 52)
for name, m in models_r.items():
    t0 = time.time()
    m.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    yp = m.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, yp))
    r2   = r2_score(y_te, yp)
    print(f"{name:20s}  {rmse:8.4f}  {r2:8.4f}  {elapsed:7.2f}s")

Dataset: 20,640 rows × 8 features

Model                     RMSE        R²      Time
----------------------------------------------------
sklearn GBM             0.4774    0.8261    29.60s
XGBoost                 0.4497    0.8457     0.81s


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM                0.4338    0.8564     0.72s
RandomForest            0.5040    0.8062    45.00s


#Insights:
---
>On a real dataset the differences are dramatic. LightGBM is 63x faster than sklearn GBM with BETTER accuracy. This is why professional ML engineers reach for XGBoost or LightGBM immediately on tabular problems — sklearn's GBM is for learning the concept only.